# Deep Learning HW2
##  Environment Setup

To ensure reproducibility, we recommend creating a dedicated conda environment before running this notebook.

### Step 1. Create a new conda environment

```bash
conda create -n dl_hw2 python=3.10 -y
conda activate dl_hw2
```
### Step 2. Install PyTorch

First, try installing the GPU version. Make sure to install the version that matches your CUDA version.
```bash
pip install --upgrade pip
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

Run the commands below to verify that PyTorch GPU version and the required packages have been installed correctly.
```bash
python -c "import sys; print(sys.executable)"
python -c "import torch; print(torch.__version__)"
python -c "import torch; print(torch.cuda.is_available())"
python -c "import numpy; print(numpy.__version__)"
```

If you do not have access to a GPU, install the CPU version instead.
```bash
pip install --upgrade pip
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
```

Run the commands below to verify that PyTorch CPU version and the required packages have been installed correctly.
```bash
python -c "import sys; print(sys.executable)"
python -c "import torch; print(torch.__version__)"
python -c "import numpy; print(numpy.__version__)"
```

##### We strongly recommend using a GPU environment. If you do not have access to a GPU, you may use Google Colab.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torchvision.datasets import OxfordIIITPet
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## Image Segmentation with U-net

#### In this part, you will implement and train a U-Net model for binary image segmentation.

For segmentation, we use the Oxford-IIIT Pet dataset.

Downloading the dataset may take some time during the first run.

The segmentation task is formulated as a binary classification problem at the pixel level.

In [ ]:
import torch
from torchvision.datasets import OxfordIIITPet
import torchvision.transforms as transforms
from torchvision.transforms import functional as TF
from PIL import Image
import matplotlib.pyplot as plt


# ============================================================
# 1. Custom transform for binary segmentation mask
# ============================================================

class BinaryMaskTransform:
    def __init__(self, size=(128, 128)):
        self.size = size

    def __call__(self, mask):
        # Resize mask using nearest-neighbor interpolation
        mask = TF.resize(mask, self.size, interpolation=Image.NEAREST)

        # Convert PIL image to tensor without normalization
        mask = torch.as_tensor(TF.pil_to_tensor(mask), dtype=torch.long)
        mask = (mask == 1).float()

        return mask


# ============================================================
# 2. Image / mask transforms
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

mask_transform = BinaryMaskTransform(size=(128, 128))


# ============================================================
# 3. Dataset
# ============================================================

train_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="segmentation",
    download=True,
    transform=image_transform,
    target_transform=mask_transform
)

test_dataset = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="segmentation",
    download=True,
    transform=image_transform,
    target_transform=mask_transform
)


# ============================================================
# 4. DataLoader
# ============================================================

batch_size = 8  # You may change the batch size based on your hardware constraints.

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)


# ============================================================
# 5. Sanity check
# ============================================================

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)   # expected: (B, 3, 128, 128)
print("Mask batch shape:", masks.shape)     # expected: (B, 1, 128, 128)
print("Mask unique values:", torch.unique(masks))  # expected: tensor([0., 1.])


# ============================================================
# 6. Visualization
# ============================================================

def show_samples(images, masks, num_samples=4):
    plt.figure(figsize=(8, 2 * num_samples))

    for i in range(num_samples):
        # image
        plt.subplot(num_samples, 2, 2 * i + 1)
        img = images[i].permute(1, 2, 0).cpu().numpy()
        plt.imshow(img)
        plt.title("Input Image")
        plt.axis("off")

        # mask
        plt.subplot(num_samples, 2, 2 * i + 2)
        mask = masks[i].squeeze().cpu().numpy()
        plt.imshow(mask, cmap="gray")
        plt.title("Binary Mask")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_samples(images, masks, num_samples=4)

In [ ]:
# ============================================================
# 1. Building Blocks
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # TODO:
        # MaxPool2d(2) -> DoubleConv

        self.block = None

    def forward(self, x):
        # TODO:
        return x


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # TODO:
        # Define:
        # 1) ConvTranspose2d for upsampling
        # 2) DoubleConv

        self.up = None
        self.conv = None

    def forward(self, x1, x2):
        # TODO:
        # x1: decoder feature
        # x2: encoder feature (skip connection)
        #
        # Hint:
        # Upsample x1, then concatenate it with x2 along the channel dimension (dim=1),
        # and apply the convolution block.

        return x1



# ============================================================
# 2. U-Net
# ============================================================

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO:
        # Define the U-Net architecture.
        #
        # Suggested channels:
        # 3 -> 16 -> 32 -> 64 -> 128 (encoder)
        # 128 -> 64 -> 32 -> 16 (decoder)
        # Final: 16 -> 1

        self.inc = None
        self.down1 = None
        self.down2 = None
        self.down3 = None

        self.up1 = None
        self.up2 = None
        self.up3 = None

        self.outc = None

    def forward(self, x):
        # TODO:
        # Implement forward pass with skip connections.
        #
        # Hint:
        # Save encoder features before downsampling,
        # and reuse them in the decoder.

        return x

In [ ]:
model = UNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30

# ============================================================
# 3. Training
# ============================================================

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device).float()

        # TODO:
        # Perform one training step and update total_loss

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.4f}")


# ============================================================
# 4. Evaluation
# ============================================================

def pixel_accuracy(preds, targets, eps=1e-8):
    preds = preds.float()
    targets = targets.float()

    # TODO: implement pixel accuracy
    return 


def dice_score(preds, targets, eps=1e-8):
    preds = preds.float()
    targets = targets.float()

    # TODO: implement dice score
    return 


def binary_miou(preds, targets, eps=1e-8):
    preds = preds.float()
    targets = targets.float()

    # TODO: implement binary mIoU
    return 


model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device).float()
        
        # TODO:
        # 1. Forward pass (logits)
        # 2. Apply sigmoid
        # 3. Convert to binary predictions (threshold = 0.5)

        pass


# TODO:
# 1. Concatenate all predictions and targets
# 2. Compute metrics for test dataset

mean_pixel_acc = 
mean_dice = 
mean_miou = 

print(f"Pixel Accuracy: {mean_pixel_acc:.4f}")
print(f"Dice Score: {mean_dice:.4f}")
print(f"mIoU: {mean_miou:.4f}")

In [ ]:
# ============================================================
# 5. Visualization
# ============================================================

def visualize_predictions(model, dataloader, device, num_examples=5):
    model.eval()
    shown = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device).float()

            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            for i in range(images.size(0)):
                if shown >= num_examples:
                    return

                image = images[i].cpu().permute(1, 2, 0).numpy()
                mask = masks[i].cpu().squeeze().numpy()
                pred = preds[i].cpu().squeeze().numpy()

                plt.figure(figsize=(9, 3))

                plt.subplot(1, 3, 1)
                plt.imshow(image)
                plt.title("Input Image")
                plt.axis("off")

                plt.subplot(1, 3, 2)
                plt.imshow(mask, cmap="gray")
                plt.title("Ground Truth")
                plt.axis("off")

                plt.subplot(1, 3, 3)
                plt.imshow(pred, cmap="gray")
                plt.title("Prediction")
                plt.axis("off")

                plt.tight_layout()
                plt.show()

                shown += 1

visualize_predictions(model, test_loader, device, num_examples=20)

## Name Classification with RNN

#### In this part, you will implement and train a RNN for sequence classification.

For the RNN task, we use the name classification dataset provided in the PyTorch tutorials.

The task is formulated as a sequence classification problem, where each input name is represented as a sequence of characters and the model predicts its corresponding category.

In [ ]:
import os
import zipfile
import urllib.request
import glob
import string
import unicodedata

url = "https://download.pytorch.org/tutorial/data.zip"
filename = "rnn_data.zip"

if not os.path.exists(filename):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete.")

if not os.path.exists("rnn_data"):
    print("Extracting dataset...")
    with zipfile.ZipFile(filename, "r") as zip_ref:
        zip_ref.extractall("rnn_data")
    print("Extraction complete.")

In [ ]:
# ============================================================
# 1. Load dataset (text files → category-wise names)
# ============================================================

data_dir = "rnn_data/data/names"

category_lines = {}   # dict: {category: [names]}
all_categories = []   # list of category names

for filename in glob.glob(os.path.join(data_dir, "*.txt")):
    category = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(category)

    with open(filename, encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        category_lines[category] = lines

all_categories = sorted(all_categories)

print("Number of categories:", len(all_categories))
print("Categories:", all_categories)


# ============================================================
# 2. Convert unicode strings to ASCII
# ============================================================

def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Apply conversion to all names
for category in all_categories:
    category_lines[category] = [
        unicode_to_ascii(name) for name in category_lines[category]
    ]


# ============================================================
# 3. Build character vocabulary
# ============================================================

all_letters = string.ascii_letters + " .,;'-"
PAD_TOKEN = "<PAD>"

char2idx = {PAD_TOKEN: 0}  # padding index = 0
for i, ch in enumerate(all_letters, start=1):
    char2idx[ch] = i

idx2char = {i: ch for ch, i in char2idx.items()}

n_chars = len(char2idx)
print("Number of characters:", n_chars)
print("Characters:", char2idx)


# ============================================================
# 4. Build label mapping (category → index)
# ============================================================

label2idx = {label: i for i, label in enumerate(all_categories)}
idx2label = {i: label for label, i in label2idx.items()}

# ============================================================
# 5. Convert dataset into (name, label) pairs
# ============================================================

data = []

for category in all_categories:
    for name in category_lines[category]:
        if len(name) > 0:
            data.append((name, category))

print("Total samples:", len(data))
print(data[:5])


# ============================================================
# 6. Train / Test split
# ============================================================
random.shuffle(data)

split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

print("Train:", len(train_data))
print("Test:", len(test_data))


# ============================================================
# 7. Encode names into sequences of character indices
# ============================================================

def encode_name(name):
    return [char2idx[ch] for ch in name if ch in char2idx]


# ============================================================
# 8. Dataset class
# ============================================================

class NameDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        name, label = self.data[idx]

        name_ids = encode_name(name)      # sequence of indices
        label_id = label2idx[label]       # class index

        return (
            torch.tensor(name_ids, dtype=torch.long),
            torch.tensor(label_id, dtype=torch.long)
        )


# ============================================================
# 9. Collate function (padding variable-length sequences)
# ============================================================

def collate_fn(batch):
    names, labels = zip(*batch)

    lengths = [len(n) for n in names]
    max_len = max(lengths)

    padded = []
    for n in names:
        pad_len = max_len - len(n)
        padded.append(torch.cat([n, torch.zeros(pad_len, dtype=torch.long)]))

    padded = torch.stack(padded)   # (B, T)
    labels = torch.stack(labels)   # (B,)
    lengths = torch.tensor(lengths)

    return padded, labels, lengths


# ============================================================
# 10. DataLoader
# ============================================================

train_dataset = NameDataset(train_data)
test_dataset = NameDataset(test_data)

batch_size = 32  # You may change the batch size based on your hardware constraints.
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)


# ============================================================
# 11. Sanity check (check batch shapes)
# ============================================================

names, labels, lengths = next(iter(train_loader))

print("Names:", names.shape)     # (B, T)
print("Labels:", labels.shape)   # (B,)
print("Lengths:", lengths.shape) # (B,)

In [ ]:
# ============================================================
# 1. RNN Model
# ============================================================

class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.hidden_size = hidden_size

        # TODO:
        # Define trainable parameters for:
        # x_t -> h_t
        # h_{t-1} -> h_t
        # h_t -> output

        self.W_xh = None   # shape: (input_size, hidden_size)
        self.W_hh = None   # shape: (hidden_size, hidden_size)
        self.b_h  = None   # shape: (hidden_size,)

        self.W_hy = None   # shape: (hidden_size, num_classes)
        self.b_y  = None   # shape: (num_classes,)

    def forward(self, x, lengths):
        """
        x: (B, T) character indices
        lengths: (B,)
        """

        # TODO:
        # 1. Convert x to one-hot vectors: (B, T, input_size)
        # 2. Initialize hidden state h0 with zeros: (B, hidden_size)
        # 3. For each time step t:
        #       h_t = tanh(x_t W_xh + h_{t-1} W_hh + b_h)
        # 4. Use the last valid hidden state for each sequence
        # 5. Compute logits: h_T W_hy + b_y

        return x
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.W_xh)
        nn.init.xavier_uniform_(self.W_hh)
        nn.init.xavier_uniform_(self.W_hy)
        nn.init.zeros_(self.b_h)
        nn.init.zeros_(self.b_y)

In [ ]:
model = RNNClassifier(
    input_size=n_chars,
    hidden_size=128,
    num_classes=len(all_categories)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 40

# ============================================================
# 3. Training
# ============================================================

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for names, labels, lengths in train_loader:
        names = names.to(device)
        labels = labels.to(device)
        lengths = lengths.to(device)

        # TODO:
        # 1. Forward pass
        # 2. Compute loss
        # 3. Backward pass
        # 4. Update parameters
        # 5. Compute accuracy

    avg_loss = total_loss / len(train_loader)
    acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")


# ============================================================
# 4. Evaluation
# ============================================================

def macro_f1(preds, targets, eps=1e-8):
    preds = preds.float()
    targets = targets.float()

    # TODO: implement macro-F1
    return

model.eval()

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for names, labels, lengths in test_loader:
        names = names.to(device)
        labels = labels.to(device)

        # TODO:
        # 1. Forward pass
        # 2. Get predicted labels
        # Hint : Store predictions and labels across all batches for F1 computation


# Accuracy
test_acc = 100 * correct / total

# TODO:
# Compute Macro-F1 using macro_f1()
mean_f1 = 

print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Macro-F1: {mean_f1:.4f}")

In [ ]:
# ============================================================
# 5. Prediction (Example outputs)
# ============================================================

def decode_name(name_tensor):
    chars = []
    for idx in name_tensor:
        idx = idx.item()
        if idx == 0:  # PAD
            continue
        chars.append(idx2char[idx])
    return "".join(chars)


def show_predictions(model, dataloader, device, num_samples=10):
    model.eval()

    shown = 0

    with torch.no_grad():
        for names, labels, lengths in dataloader:
            names = names.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)

            outputs = model(names, lengths)
            _, preds = torch.max(outputs, 1)

            batch_size = names.size(0)

            for i in range(batch_size):
                if shown >= num_samples:
                    return

                name_str = decode_name(names[i].cpu())
                true_label = idx2label[labels[i].item()]
                pred_label = idx2label[preds[i].item()]

                print(f"{name_str:15s} | GT: {true_label:10s} | Pred: {pred_label}")

                shown += 1


show_predictions(model, test_loader, device, num_samples=10)

In some cases, the test accuracy may be relatively high, while the Macro-F1 score is significantly lower.

**Question:**  
Explain why this phenomenon occurs.  